# From Stars to Code — An Introduction to Exoplanet Detection

**A lecture notebook for the ExoHunter study group**

This notebook teaches the science behind exoplanet transit detection
and shows how each concept is implemented in the ExoHunter codebase.
No astronomy background is required — we start from scratch.

### How to use this notebook

- **Read** the markdown cells for theory and context
- **Run** the code cells to see interactive visualizations
- **Experiment** by changing parameters and observing the effect
- Each section connects a scientific concept to its code implementation

```bash
pip install -e ".[dev]"
```

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# We'll import ExoHunter modules as needed in each section
print("Ready.")

---

## 1. Why exoplanets matter

For most of human history, we had no idea whether planets existed
beyond our Solar System. The first confirmed exoplanet around a
Sun-like star was discovered in **1995** (51 Pegasi b). Since then,
the count has exploded — over **5,700 confirmed exoplanets** as of 2025.

The driving question: **are we alone?** To answer it, we need to find
rocky planets in the habitable zone — the region around a star where
liquid water could exist on the surface.

### Discovery methods

| Method | How it works | Planets found |
|--------|-------------|---------------|
| **Transit** | Planet blocks starlight (tiny dip in brightness) | ~4,200 (75%) |
| Radial velocity | Star wobbles due to planet's gravity | ~1,100 |
| Direct imaging | Photograph the planet directly | ~70 |
| Microlensing | Planet bends background starlight | ~200 |

The transit method dominates because space telescopes like **Kepler** and
**TESS** can monitor hundreds of thousands of stars simultaneously.
ExoHunter implements this method.

In [ ]:
# Visualization: exoplanet discovery timeline
# Each bar represents cumulative confirmed planets per year

years = list(range(1995, 2026))
# Approximate cumulative counts (source: NASA Exoplanet Archive)
cumulative = [
    1, 6, 9, 12, 20, 29, 47, 82, 106, 126, 154, 193, 247, 310, 358,
    461, 573, 714, 862, 991, 1729, 2973, 3440, 3706, 3805, 4046,
    4164, 4401, 4776, 5197, 5700
]

# Animated bar chart — bars grow year by year
frames = []
for i in range(len(years)):
    frames.append(go.Frame(
        data=[go.Bar(
            x=years[:i+1], y=cumulative[:i+1],
            marker_color=["gold" if y == years[i] else "deepskyblue" for y in years[:i+1]],
            text=[str(c) if j == i else "" for j, c in enumerate(cumulative[:i+1])],
            textposition="outside", textfont=dict(color="white", size=11),
        )],
        name=str(years[i]),
        layout=go.Layout(title=f"Confirmed Exoplanets — {years[i]} ({cumulative[i]:,} total)"),
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        title=f"Confirmed Exoplanets — 1995 ({cumulative[0]} total)",
        xaxis=dict(title="Year", range=[1993, 2027]),
        yaxis=dict(title="Cumulative planets", range=[0, 6500]),
        height=450,
        updatemenus=[dict(
            type="buttons", showactive=False, y=0, x=0.5, xanchor="center",
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 200, "redraw": True}}]),
            ],
        )],
    ),
)
fig.show()

---

## 2. The transit method — how a dip reveals a planet

When a planet passes in front of its star (a **transit**), it blocks
a fraction of the star's light. The amount of light blocked depends
on the planet's size relative to the star:

$$\text{depth} = \left(\frac{R_{\text{planet}}}{R_{\text{star}}}\right)^2$$

For example:
- **Jupiter** around the Sun: depth = (0.1)² = **1%** (easy to detect)
- **Earth** around the Sun: depth = (0.009)² = **0.008%** (very hard!)
- **Earth** around an M-dwarf (small star): depth = (0.009/0.003)² = **0.09%** (easier!)

This is why TESS focuses on bright, nearby M-dwarfs — small stars
make small planets easier to find.

### Try it yourself

The interactive visualization below lets you change the planet radius
and see how the transit depth changes. Drag the slider and watch the
transit dip deepen or shrink.

In [ ]:
# Interactive transit depth calculator
# Slider controls Rp/Rs, transit dip updates live

from exohunter.detection.model import transit_model

time_grid = np.linspace(-0.15, 0.15, 500)  # phase units

# Planet sizes to sweep: from Mars-sized to Jupiter-sized
rp_rs_values = np.linspace(0.02, 0.15, 40)

# Reference planet sizes (Rp/Rs for a solar-type star)
references = {
    "Mars": 0.005, "Earth": 0.009, "Neptune": 0.035,
    "Saturn": 0.083, "Jupiter": 0.10,
}

frames = []
for rp_rs in rp_rs_values:
    depth = rp_rs ** 2
    duration = 0.04  # fixed for visual clarity
    model = transit_model(
        time_grid, period=1.0, epoch=0.0,
        duration=duration, depth=depth, ingress_fraction=0.15,
    )

    # Find the closest named planet
    closest_name = min(references, key=lambda k: abs(references[k] - rp_rs))
    closest_dist = abs(references[closest_name] - rp_rs)
    label = f" (near {closest_name})" if closest_dist < 0.01 else ""

    frames.append(go.Frame(
        data=[go.Scatter(
            x=time_grid, y=model, mode="lines",
            line=dict(color="deepskyblue", width=3),
            fill="tozeroy", fillcolor="rgba(0,100,255,0.1)",
        )],
        name=f"{rp_rs:.3f}",
        layout=go.Layout(
            title=f"Rp/Rs = {rp_rs:.3f} → depth = {depth*100:.3f}%{label}"
        ),
    ))

sliders = [dict(
    active=10,
    steps=[dict(
        method="animate",
        args=[[f"{r:.3f}"], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
        label=f"{r:.2f}",
    ) for r in rp_rs_values],
    currentvalue=dict(prefix="Rp/Rs = "),
    pad=dict(t=50),
)]

fig = go.Figure(
    data=frames[10].data,
    frames=frames,
    layout=go.Layout(
        template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        title=f"Rp/Rs = {rp_rs_values[10]:.3f} → depth = {(rp_rs_values[10]**2)*100:.3f}%",
        xaxis=dict(title="Orbital Phase", range=[-0.15, 0.15]),
        yaxis=dict(title="Normalized Flux", range=[0.975, 1.003]),
        height=450,
        sliders=sliders,
    ),
)
fig.show()

**In the code**: ExoHunter's `transit_model()` function in
`exohunter/detection/model.py` generates the trapezoidal shape you see
above. It takes period, epoch, duration, and depth as inputs and
produces a flux array — the same model used for subtraction in the
multi-planet search.

---

## 3. TESS — our telescope

The **Transiting Exoplanet Survey Satellite** (TESS) is a NASA space
telescope launched in 2018. It divides the sky into **sectors** (26 per
hemisphere) and stares at each one for ~27 days.

Key facts:
- **Cadence**: one brightness measurement every **2 minutes**
- **Targets**: pre-selected bright stars (SPOC pipeline)
- **Archive**: all data is public at MAST (Mikulski Archive)
- **Time system**: BTJD (Barycentric TESS Julian Date)

A single 27-day sector produces ~20,000 data points per star.
Stars near the ecliptic poles are observed in **multiple sectors**,
giving baselines of hundreds of days — crucial for finding
long-period planets.

**In the code**: `exohunter/ingestion/downloader.py` handles
all interaction with MAST. The function `download_lightcurve()` fetches
all available sectors for a target and stitches them into one continuous
light curve. Results are cached as FITS files to avoid re-downloading.

---

## 4. Cleaning the signal — preprocessing

Raw telescope data is **noisy**. Before searching for transits, we
must remove artefacts that would confuse the detection algorithm.

### Sources of noise

| Source | What it looks like | How we fix it |
|--------|--------------------|---------------|
| **NaN values** | Missing data (gaps) | Remove cadences |
| **Cosmic rays** | Single-point spikes | Sigma-clipping (5-sigma) |
| **Instrumental units** | Flux in electrons/s, varies by star | Normalize to median = 1.0 |
| **Stellar variability** | Slow undulations (starspots, rotation) | Savitzky-Golay filter |

### The ExoHunter preprocessing pipeline

```
Raw flux → remove NaN → sigma-clip → normalize → flatten → CDPP estimate
```

The visualization below shows each step's effect on real data.
Use the buttons to toggle each step on/off.

In [ ]:
# Step-by-step preprocessing visualization
# We generate a synthetic light curve with all noise types, then clean it

from lightkurve import LightCurve

rng = np.random.default_rng(42)
n = 8000
time = np.linspace(0, 27.4, n)  # one TESS sector

# Build a "dirty" light curve with multiple noise sources
flux_clean = np.ones(n)

# Inject a transit (P=5d, depth=1%)
phase = (time % 5.0) / 5.0
in_transit = np.abs(phase - 0.5) < 0.008
flux_clean[in_transit] -= 0.01

# Add noise layers
stellar_var = 0.005 * np.sin(2 * np.pi * time / 3.7)  # starspot rotation
photon_noise = rng.normal(0, 0.002, n)
cosmic_rays = np.zeros(n)
cosmic_rays[rng.choice(n, 15)] = rng.uniform(0.02, 0.08, 15)  # spikes

flux_raw = flux_clean + stellar_var + photon_noise + cosmic_rays
flux_raw[500:520] = np.nan  # data gap

# Apply each step
from exohunter.preprocessing.clean import remove_nans, remove_outliers
from exohunter.preprocessing.normalize import normalize_lightcurve
from exohunter.preprocessing.detrend import flatten_lightcurve

lc0 = LightCurve(time=time, flux=flux_raw)
lc1 = remove_nans(lc0)
lc2 = remove_outliers(lc1, sigma=5.0)
lc3 = normalize_lightcurve(lc2)
lc4 = flatten_lightcurve(lc3)

steps = [
    ("Raw (all noise)", time, flux_raw, "gray"),
    ("After NaN removal", lc1.time.value, lc1.flux.value, "#ff6b6b"),
    ("After sigma-clipping", lc2.time.value, lc2.flux.value, "#ffd93d"),
    ("After normalization", lc3.time.value, lc3.flux.value, "#4ecdc4"),
    ("After flattening", lc4.time.value, lc4.flux.value, "deepskyblue"),
]

# Create frames for each step
frames = []
for name, t, f, color in steps:
    frames.append(go.Frame(
        data=[go.Scattergl(
            x=t, y=f, mode="markers",
            marker=dict(size=2, color=color, opacity=0.6),
        )],
        name=name,
        layout=go.Layout(title=f"Preprocessing: {name}"),
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        title="Preprocessing: Raw (all noise)",
        xaxis_title="Time (days)", yaxis_title="Flux",
        height=400,
        updatemenus=[dict(
            type="buttons", direction="right",
            showactive=True, x=0.0, y=1.15,
            buttons=[dict(
                label=name.split("(")[0].strip() if "(" in name else name[:20],
                method="animate",
                args=[[name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
            ) for name, _, _, _ in steps],
        )],
    ),
)
fig.show()

**In the code**: The full pipeline is in `exohunter/preprocessing/pipeline.py`.
Each step has its own module (`clean.py`, `normalize.py`, `detrend.py`)
so students can study and modify them independently.

Key parameter: the Savitzky-Golay flatten window is **1001 cadences**
(~33 hours). This removes stellar variability on timescales > 1 day
while preserving transit dips lasting 1-13 hours.

---

## 5. Finding the signal — the BLS algorithm

### The problem

We have ~20,000 flux measurements. Somewhere in there, a planet
causes periodic dips of 0.01-1%. We don't know the period, the phase,
the duration, or the depth. We must search over all possibilities.

### The solution: Box Least Squares (BLS)

For each trial period P:
1. **Phase-fold** the data: wrap all points onto a single orbit
2. **Slide a box** across the phase: compute the mean flux inside
   and outside the box
3. The **power** measures how significant the depth is:

$$\text{power} = \frac{n_{\text{in}} \cdot n_{\text{out}} \cdot \delta^2}{n_{\text{total}}}$$

where $\delta$ = (mean\_out - mean\_in) is the transit depth.

The period with the highest power is the best transit candidate.

### Why is it fast?

A naive implementation tests every data point at every phase position
— $O(n_{\text{periods}} \times n_{\text{bins}} \times n_{\text{data}})$,
which is trillions of operations.

ExoHunter's Numba implementation uses **binned prefix sums**: bin the
phase-folded data into 300 bins, then slide the box using cumulative
sums. This reduces complexity to
$O(n_{\text{periods}} \times (n_{\text{data}} + n_{\text{bins}}))$
— a ~1000x speedup.

### See it in action

The animation below demonstrates phase-folding. At the wrong period,
data points scatter randomly. At the correct period, the transit dip
emerges from the noise.

In [ ]:
# BLS intuition: phase-fold at wrong vs right period

# Create a synthetic transit light curve
rng2 = np.random.default_rng(99)
t_syn = np.linspace(0, 90, 15000)
f_syn = np.ones_like(t_syn)
true_period = 6.0
phase_syn = (t_syn % true_period) / true_period
f_syn[np.abs(phase_syn - 0.5) < 0.008] -= 0.008  # 0.8% depth
f_syn += rng2.normal(0, 0.002, len(t_syn))

# Show three cases side by side: wrong, close, correct period
trial_periods = [3.7, 5.5, 6.0]
titles = ["Wrong period (3.7 d) — noise",
          "Close but wrong (5.5 d) — smeared",
          "Correct period (6.0 d) — transit!"]

fig = make_subplots(rows=1, cols=3, subplot_titles=titles)

for col, (tp, title) in enumerate(zip(trial_periods, titles), 1):
    ph = ((t_syn - 3.0) % tp) / tp
    ph = np.where(ph > 0.5, ph - 1.0, ph)

    # Bin for clarity
    from exohunter.detection.model import bin_phase_curve
    sort_idx = np.argsort(ph)
    centers, means, _ = bin_phase_curve(ph[sort_idx], f_syn[sort_idx], n_bins=150)

    fig.add_trace(go.Scattergl(
        x=ph, y=f_syn, mode="markers",
        marker=dict(size=1, color="rgba(100,149,237,0.1)"),
        showlegend=False,
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=centers, y=means, mode="lines+markers",
        marker=dict(size=3, color="deepskyblue"),
        line=dict(width=2, color="deepskyblue"),
        showlegend=False,
    ), row=1, col=col)
    fig.update_xaxes(range=[-0.15, 0.15], row=1, col=col)

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=350,
    title="Why the correct period matters — phase-folding at 3 trial periods",
)
fig.show()

**In the code**: ExoHunter provides two BLS implementations in
`exohunter/detection/bls.py`:

| Implementation | Performance | Purpose |
|---|---|---|
| `run_bls_lightkurve()` | 1.19 s | Production — uses astropy's C code |
| `run_bls_numba()` via `_bls_core()` | 0.20 s | Pedagogical — from-scratch with Numba JIT, students can read the algorithm |

Both are tested to give the same results (see `test_bls.py`).

---

## 6. Is it real? — validation

BLS finds the strongest periodic signal, but not every signal is a
planet. We apply **6 tests** to decide if a candidate is genuine.

The flowchart below shows the decision process.

In [ ]:
# Validation decision tree — visual infographic
# Simulates a candidate passing through each test

tests = [
    ("1. SNR >= 7?",           "Signal strong enough?",       True,  "#4ecdc4"),
    ("2. 0.01% < depth < 5%?", "Depth physically plausible?", True,  "#4ecdc4"),
    ("3. Duration/Period OK?",  "Consistent with orbit?",      True,  "#4ecdc4"),
    ("4. >= 3 transits?",       "Enough observations?",        True,  "#4ecdc4"),
    ("5. Box-shaped (not V)?",  "Planet, not binary?",         True,  "#ffd93d"),
    ("6. Not a harmonic?",      "Independent signal?",         True,  "#ffd93d"),
]

fig = go.Figure()

y_positions = list(range(len(tests), 0, -1))

for i, (test_name, question, passed, color) in enumerate(tests):
    y = y_positions[i]
    symbol = "PASS" if passed else "FAIL"
    critical = "CRITICAL" if i < 4 else "WARNING"

    # Test box
    fig.add_shape(type="rect", x0=0, x1=5, y0=y-0.35, y1=y+0.35,
                  fillcolor=color, opacity=0.2, line=dict(color=color, width=2))
    fig.add_annotation(x=2.5, y=y, text=f"<b>{test_name}</b><br><i>{question}</i>",
                       showarrow=False, font=dict(size=11, color="white"))

    # Result badge
    badge_color = "limegreen" if passed else "tomato"
    fig.add_shape(type="rect", x0=5.5, x1=7, y0=y-0.25, y1=y+0.25,
                  fillcolor=badge_color, opacity=0.8, line=dict(width=0))
    fig.add_annotation(x=6.25, y=y, text=f"<b>{symbol}</b>",
                       showarrow=False, font=dict(size=11, color="white"))

    # Type badge
    type_color = "#ff6b6b" if critical == "CRITICAL" else "#ffd93d"
    fig.add_annotation(x=8, y=y, text=critical,
                       showarrow=False, font=dict(size=9, color=type_color))

    # Arrow to next
    if i < len(tests) - 1:
        fig.add_annotation(x=2.5, y=y-0.35, ax=2.5, ay=y_positions[i+1]+0.35,
                           arrowhead=2, arrowsize=1.5, arrowwidth=2,
                           arrowcolor="gray", showarrow=True, text="")

# Final verdict
fig.add_shape(type="rect", x0=1, x1=4, y0=-0.35, y1=0.35,
              fillcolor="limegreen", opacity=0.3, line=dict(color="limegreen", width=3))
fig.add_annotation(x=2.5, y=0, text="<b>VALIDATED</b>",
                   showarrow=False, font=dict(size=14, color="limegreen"))
fig.add_annotation(x=2.5, y=y_positions[-1]-0.35, ax=2.5, ay=0.35,
                   arrowhead=2, arrowsize=1.5, arrowwidth=2,
                   arrowcolor="limegreen", showarrow=True, text="")

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    title="Validation Pipeline — 6 Tests Every Candidate Must Pass",
    xaxis=dict(visible=False, range=[-0.5, 9]),
    yaxis=dict(visible=False, range=[-1, 7.5]),
    height=550, width=800,
    showlegend=False,
)
fig.show()

Tests 1-4 are **hard requirements** — failing any one rejects the
candidate. Tests 5-6 produce warnings that reduce the candidate's
priority score but don't reject it outright.

**In the code**: `exohunter/detection/validator.py` implements all 6
tests. The thresholds are in `exohunter/config.py` — students can
modify them and see how the acceptance rate changes.

---

## 7. What is it? — cross-matching and classification

After validation, we compare each candidate against the
**ExoFOP-TESS TOI catalog** (~7,900 known objects of interest)
to classify it:

| Class | Color | Meaning |
|-------|-------|---------|
| **NEW_CANDIDATE** | Bright green | Not in any catalog — potential discovery! |
| **KNOWN_MATCH** | Blue | Re-detection of a known planet |
| **KNOWN_TOI** | Yellow | Known star, different period |
| **HARMONIC** | Orange | Period alias of a known signal |

`NEW_CANDIDATE` is the exciting one — it means we found something
that nobody has catalogued yet.

**In the code**: `exohunter/catalog/crossmatch.py` queries the NASA
Exoplanet Archive via TAP API (live), with fallback to a local CSV
and a built-in reference table. The catalog is cached for 48 hours
before auto-refreshing.

---

## 8. How confident? — ML classification

The rule-based validator catches obvious false positives, but
borderline cases need a more nuanced approach. ExoHunter includes a
**Random Forest classifier** trained on ~7,600 labeled examples from
the Kepler mission.

### Features used

The classifier takes 10 features per candidate and predicts one of
three classes: `planet`, `eclipsing_binary`, or `false_positive`.

In [ ]:
# Infographic: feature importance (simulated from typical trained model)

features = [
    "SNR", "depth (log)", "impact parameter", "duration/period",
    "stellar radius", "period", "stellar Teff", "duration",
    "stellar logg", "depth",
]
importances = [0.22, 0.18, 0.14, 0.11, 0.09, 0.08, 0.06, 0.05, 0.04, 0.03]

colors = ["#00ff88" if imp > 0.10 else "deepskyblue" if imp > 0.05 else "gray"
          for imp in importances]

fig = go.Figure(go.Bar(
    y=features[::-1], x=importances[::-1],
    orientation="h",
    marker_color=colors[::-1],
    text=[f"{imp:.0%}" for imp in importances[::-1]],
    textposition="outside", textfont=dict(color="white"),
))

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    title="Random Forest Feature Importances — What Matters Most?",
    xaxis_title="Importance", height=400,
    xaxis=dict(range=[0, 0.30]),
)
fig.show()

**SNR** (signal strength) and **depth** (transit size) are the strongest
predictors — this makes physical sense. The **impact parameter**
(how centrally the planet crosses the star) is third, because
grazing transits are more often false positives.

**In the code**: `exohunter/classification/` contains the full ML
pipeline: dataset download, feature extraction, model training, and
prediction. See [ML_GUIDE.md](../ML_GUIDE.md) for the walkthrough.

---

## 9. Putting it all together — the ExoHunter pipeline

### Architecture

```
MAST Archive (NASA)
       |
       v
  INGESTION         ThreadPoolExecutor (8 threads)
  Download + cache   I/O-bound → threads release the GIL
       |
       v
  PREPROCESSING      ProcessPoolExecutor (N-1 cores)
  Clean + flatten    CPU-bound → processes bypass the GIL
       |
       v
  DETECTION          Numba @njit(parallel=True)
  BLS + validate     CPU-bound → JIT-compiled SIMD parallelism
       |
       v
  CATALOG            Cross-match + ML classification
  Score + rank       4-tier classification + Random Forest
       |
       v
  DASHBOARD          Plotly Dash (DARKLY theme)
  Visualize          Sky map, light curve, phase diagram
```

### Concurrency model

A key design decision: **different stages use different parallelism
strategies** because they have different bottlenecks.

| Stage | Bottleneck | Strategy | Why not the other? |
|-------|-----------|----------|--------------------|
| Download | Network I/O | **Threads** | Processes would waste memory duplicating the Python interpreter |
| Preprocessing | CPU (SG filter) | **Processes** | Threads can't run Python CPU code in parallel (GIL) |
| BLS | CPU (nested loops) | **Numba JIT** | Compiles to machine code, bypasses Python entirely |

This is a central lesson for the concurrent programming course:
**there is no single best parallelism strategy** — you must match
the strategy to the bottleneck.

### Where to go from here

| Notebook | What you'll do |
|----------|----------------|
| **01_exploratory.ipynb** | Guided walkthrough of the full pipeline on TOI-700 |
| **02_student_exercises.ipynb** | Analyze L 98-59 yourself — find the planets, measure detection limits |

| Script | What it does |
|--------|-------------|
| `scripts/run_dashboard.py` | Launch the interactive dashboard |
| `scripts/run_batch.py --sector 56` | Process an entire TESS sector |
| `scripts/inspect_candidate.py` | Generate a 4-panel diagnostic report |
| `scripts/train_classifier.py` | Train the ML classifier on Kepler data |